# Preprocesamiento de datos

En este notebook se describe el proceso de preprocesamiento aplicado al conjunto de datos utilizado para la predicción de readmisión hospitalaria a 30 días. Se escogio el umbral de 30 días porque en la mayoria de papers se usaba este máximo.

Las principales tareas que se realizarán en esta fase son:

- Limpieza de datos
- Tratamiento de valores faltantes
- Codificación de variables categóricas
- Transformación de variables
- Preparación del dataset final para el modelado

Para ello se creo el scrip `preprocessing.py`este se encarga de hacer una limpieza general. La idea es ir mirando si lo hace de manera correcta ajustando las funciones para que al final se puedan usar estas de manera definitiva para tener un mejor pipeline. El notebook se utiliza como apoyo para comprobar paso a paso que dichas funciones realizan correctamente el procesamiento y para justificar las decisiones adoptadas durante esta fase.

En el pipeline de preprocesamiento se utilizan tres tablas de MIMIC-IV: `patients`, `admissions` y `diagnoses_icd`. A partir de ellas se construye el dataset base que combina información demográfica del paciente, datos de cada hospitalización y una variable clínica derivada: el número de diagnósticos por ingreso (`n_diagnoses`), obtenida a partir de la tabla de diagnósticos.

Las tablas de `procedures` y `prescriptions` no se incluyen en esta versión del pipeline. Aunque fueron analizadas en el EDA, en este caso al ser tablsa tan grandes  dificultaría el procesamiento en el entorno disponible.

El proceso de preprocesamiento se dividió en dos etapas principales.

En la **primera etapa** (`run_preprocessing_part1`) se realizó una limpieza estructural del dataset: eliminación de duplicados, conversión de variables temporales a formato datetime, generación de `length_of_stay` (duración de la estancia en días) y filtrado de registros con duración negativa. Además, se integró la información de las tres tablas y se añadió la variable `n_diagnoses`, que recoge el número de diagnósticos registrados por ingreso como indicador (referencia de otros papers).

En la **segunda etapa** (`run_preprocessing_part2`) se realizaron las transformaciones orientadas al modelado:

- **Filtrado de fallecidos durante el ingreso**: los pacientes con `hospital_expire_flag=1` se excluyen antes de construir la variable objetivo, ya que no pueden tener readmisión y sesgarían la distribución de la clase.
- **Creación de la variable objetivo** `readmission_30_days`: indica si el paciente reingresó en los 30 días siguientes al alta. Se incorpora también `previous_admissions` (número de ingresos previos).
- **Cálculo de la edad real en el ingreso** (`age_at_admission`): MIMIC-IV desplaza las fechas por privacidad, por lo que `anchor_age` no corresponde a la edad en cada ingreso concreto. Se calcula como `anchor_age + (año_de_admisión - anchor_year)`.
- **Eliminación de variables no adecuadas**: identificadores, variables temporales ya resumidas, variables con riesgo de fuga de información (`deathtime`, `dod`) y columnas auxiliares.
- **Imputación de categóricas** con `Unknown` para no perder registros.
- **Agrupación de categorías poco frecuentes** en `Other` para `race` y `language` (umbral del 1%), reduciendo la cardinalidad antes del encoding.
- **Codificación one-hot** con `drop_first=True` para evitar la trampa de variables dummy.

# 1. Limpieza estructural y construcción del dataset base

En esta primera parte se comprueba el funcionamiento de `run_preprocessing_part1`, responsable de generar un dataset limpio a partir de las tablas originales y de crear variables derivadas relevantes para el análisis posterior.

In [35]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.data.preprocessing import  run_preprocessing_part1

df = run_preprocessing_part1()

df.head()

Loading datasets...
Loading patients...
Loading admissions...
Loading diagnoses...
Cleaning datasets...
Merging datasets...
Adding diagnosis features...
Computing previous admissions...
Saving interim dataset...
Preprocessing completed.


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,edouttime,hospital_expire_flag,length_of_stay,gender,anchor_age,anchor_year,anchor_year_group,dod,n_diagnoses,previous_admissions
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,2180-05-06 23:30:00,0,0,F,52,2180,2014 - 2016,2180-09-09,8,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,2180-06-26 21:31:00,0,1,F,52,2180,2014 - 2016,2180-09-09,8,1
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,2180-07-23 14:00:00,0,2,F,52,2180,2014 - 2016,2180-09-09,13,2
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,2180-08-06 01:44:00,0,1,F,52,2180,2014 - 2016,2180-09-09,10,3
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,2160-03-04 06:26:00,0,0,F,19,2160,2008 - 2010,NaN,1,0


In [36]:
# Validación: n_diagnoses debe estar presente y sin negativos
print("n_diagnoses generado:", "n_diagnoses" in df.columns)
print(df["n_diagnoses"].describe())
print("Valores faltantes en n_diagnoses:", df["n_diagnoses"].isnull().sum())

n_diagnoses generado: True
count    545853.000000
mean         11.656307
std           7.630953
min           0.000000
25%           6.000000
50%          10.000000
75%          16.000000
max          57.000000
Name: n_diagnoses, dtype: float64
Valores faltantes en n_diagnoses: 0


In [37]:
df.shape


(545853, 24)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 545853 entries, 0 to 545852
Data columns (total 24 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            545853 non-null  int64         
 1   hadm_id               545853 non-null  int64         
 2   admittime             545853 non-null  datetime64[ns]
 3   dischtime             545853 non-null  datetime64[ns]
 4   deathtime             11689 non-null   object        
 5   admission_type        545853 non-null  object        
 6   admit_provider_id     545849 non-null  object        
 7   admission_location    545852 non-null  object        
 8   discharge_location    396100 non-null  object        
 9   insurance             536519 non-null  object        
 10  language              545082 non-null  object        
 11  marital_status        532260 non-null  object        
 12  race                  545853 non-null  object        
 13  edre

In [39]:
df.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,edouttime,hospital_expire_flag,length_of_stay,gender,anchor_age,anchor_year,anchor_year_group,dod,n_diagnoses,previous_admissions
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,2180-05-06 23:30:00,0,0,F,52,2180,2014 - 2016,2180-09-09,8,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,2180-06-26 21:31:00,0,1,F,52,2180,2014 - 2016,2180-09-09,8,1
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,2180-07-23 14:00:00,0,2,F,52,2180,2014 - 2016,2180-09-09,13,2
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,2180-08-06 01:44:00,0,1,F,52,2180,2014 - 2016,2180-09-09,10,3
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,2160-03-04 06:26:00,0,0,F,19,2160,2008 - 2010,NaN,1,0


In [40]:
df.isnull().sum().sort_values(ascending=False)

deathtime               534164
dod                     400997
edouttime               166757
edregtime               166757
discharge_location      149753
marital_status           13593
insurance                 9334
language                   771
admit_provider_id            4
admission_location           1
admission_type               0
dischtime                    0
subject_id                   0
hadm_id                      0
admittime                    0
race                         0
hospital_expire_flag         0
length_of_stay               0
anchor_age                   0
gender                       0
anchor_year                  0
anchor_year_group            0
n_diagnoses                  0
previous_admissions          0
dtype: int64

In [41]:
df.duplicated().sum()

np.int64(0)

In [42]:
df["length_of_stay"].describe()

count    545853.000000
mean          4.222706
std           7.201838
min           0.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         515.000000
Name: length_of_stay, dtype: float64

In [43]:
df["subject_id"].nunique()

223382

In [44]:
import os
os.listdir("../data/interim")

['.gitkeep', 'cleaned_dataset.csv']

In [45]:
summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "n_patients": df["subject_id"].nunique(),
    "n_admissions": df["hadm_id"].nunique()
}

summary

{'n_rows': 545853,
 'n_columns': 24,
 'n_patients': 223382,
 'n_admissions': 545853}

# 2. Preparación del conjunto final para modelado

En esta segunda parte se parte del dataset limpio `cleaned_dataset` para aplicar las transformaciones finales orientadas al modelado. Entre ellas se incluyen la eliminación de variables no relevantes, el tratamiento de valores faltantes y la codificación de variables categóricas.

Trabajar desde este dataset intermedio permite reutilizar una versión ya depurada de los datos sin repetir todo el procesamiento inicial, lo que mejora el pipeline.

## Variable objetivo y variables derivadas

La variable objetivo `readmission_30_days` se construye a partir de la secuencia temporal de ingresos de cada paciente. Para cada hospitalización se calcula el intervalo entre el alta y el siguiente ingreso del mismo paciente; si ese intervalo es menor o igual a 30 días, se considera readmisión (valor 1).

Las últimas admisiones de cada paciente se eliminan del dataset, ya que no es posible saber si hubo readmisión posterior y su inclusión introduciría sesgo.

Además se generan dos variables adicionales:
- `previous_admissions`: número de ingresos anteriores del mismo paciente, para tener una frecuencia.
- `age_at_admission`: edad real del paciente en el momento del ingreso, corrigiendo el desplazamiento temporal aplicado por MIMIC-IV.

# Eliminación de variables no adecuadas e imputación de valores categóricos

Antes de entrenar los modelos es necesario revisar qué variables deben mantenerse y cuáles conviene eliminar. En esta fase se excluyen columnas que:

- no aportan información útil para la predicción,
- presentan riesgo de fuga de información,
- resultan redundantes con otras variables ya derivadas,
- o describen eventos demasiado específicos para el objetivo del modelo.

Además, se imputan los valores faltantes en variables categóricas con una categoría explícita como `Unknown`, de forma que la ausencia de información no obligue a eliminar registros  útiles.

Algunas variables, como `deathtime`, `dod`, `edregtime` o `edouttime`, se eliminan por no resultar adecuadas para esta versión del problema predictivo. 

También se eliminan variables temporales originales como `admittime` y `dischtime` una vez que ya se ha creado la feature`length_of_stay`, que resume de forma más directa la duración del ingreso.

En el caso de variables categóricas con un número moderado de valores faltantes, como `insurance`, `marital_status` o `language`, se utiliza la categoría `Unknown` para conservar la información de cara ak modelo sin introducir imputacioness.

# Codificación de variables categóricas

Una vez tratadas las categorías faltantes, las variables de tipo objeto se transforman a formato numérico mediante codificación *one-hot*. Este enfoque permite representar variables categóricas con múltiples valores sin imponer un orden artificial entre categorías.

# Validación del pipeline implementado en `run_preprocessing_part2`


In [46]:
import sys
import os
import pandas as pd 
sys.path.append(os.path.abspath(".."))
from src.config import DATA_INTERIM
from src.data.preprocessing import run_preprocessing_part2


df =pd.read_csv(DATA_INTERIM/"cleaned_dataset.csv",
     parse_dates=["admittime","dischtime"]
     )
df.columns

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'length_of_stay',
       'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod',
       'n_diagnoses', 'previous_admissions'],
      dtype='object')

In [47]:
df_model = run_preprocessing_part2(df.copy())
print("Columnas:", len(df_model.columns))

Preparing dataset for modeling...
Filtering in-hospital deaths...
Creating readmission target variable...
Columns before processing: 26
Columns after encoding: 55
Saving processed dataset...
Model dataset created.
Columnas: 55


In [48]:
df_model[["previous_admissions","readmission_30_days"]].head(10)

,previous_admissions,readmission_30_days
0,0,0
1,1,1
2,2,1
5,0,0
8,0,0
14,0,0
16,0,0
17,1,0
18,2,0
22,0,1


In [49]:
df_model["readmission_30_days"].value_counts()
df_model.head()

,length_of_stay,n_diagnoses,previous_admissions,readmission_30_days,age_at_admission,gender_M,race_ASIAN - CHINESE,race_BLACK/AFRICAN AMERICAN,race_BLACK/CAPE VERDEAN,race_HISPANIC OR LATINO,...,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL,discharge_location_HOME,discharge_location_HOME HEALTH CARE,discharge_location_Other,discharge_location_REHAB,discharge_location_SKILLED NURSING FACILITY,discharge_location_Unknown
0,0,8,0,0,52,False,False,False,False,False,...,False,True,False,False,True,False,False,False,False,False
1,1,8,1,1,52,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,2,13,2,1,52,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
5,4,6,0,0,72,True,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False
8,0,9,0,0,55,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [50]:
len(df_model.columns)


55

In [51]:
# Validación: age_at_admission debe estar presente y en rango razonable
print("age_at_admission generado:", "age_at_admission" in df_model.columns)
print(df_model[["age_at_admission", "length_of_stay", "n_diagnoses", "previous_admissions"]].describe())

age_at_admission generado: True
       age_at_admission  length_of_stay    n_diagnoses  previous_admissions
count     315982.000000   315982.000000  315982.000000        315982.000000
mean          60.064985        4.128238      11.915343             4.659129
std           18.126838        6.891251       7.328473             9.802050
min           18.000000        0.000000       0.000000             0.000000
25%           48.000000        1.000000       6.000000             0.000000
50%           61.000000        2.000000      11.000000             2.000000
75%           74.000000        5.000000      16.000000             5.000000
max          105.000000      321.000000      57.000000           235.000000


## Resumen del preprocesamiento

El pipeline genera un dataset final con las siguientes características:

- **315.982 registros** (ingresos con alta conocida, descartados fallecidos en hospital y últimas admisiones sin seguimiento)
- **Variables numéricas**: `length_of_stay`, `age_at_admission`, `n_diagnoses`, `previous_admissions`
- **Variable objetivo**: `readmission_30_days`
- **Variables categóricas**: codificadas en one-hot tras agrupar categorías raras (`race`, `language` y `discharge_location` con umbral del 1%)
- **Total de columnas**: 55 tras el encoding

Este dataset queda guardado en `data/processed/model_dataset.csv` y es el punto de entrada para la fase de modelado.